### Importation des librairies

In [ ]:
import pandas as pd
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from tqdm import tqdm
import warnings
import numpy as np

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingRegressor


### Importation des fichiers

In [ ]:
x_train = pd.read_csv('data/x_train.csv', index_col=0)
x_test = pd.read_csv('data/x_test.csv', index_col=0)
y_train = pd.read_csv('data/y_train.csv', index_col=0)
sample_submission = pd.read_csv('data/new_output_sample.csv', index_col=0)

print("Dimensions x_train :", x_train.shape)
print("Dimensions y_train :", y_train.shape)
print("Dimensions x_test :", x_test.shape)
print("Dimensions sample_submission :", sample_submission.shape)

### Fonctions utiles

In [ ]:
# Génération du fichier de soumission
def generer_soumission(soumission, nom_sortie):
    is_valid = (soumission.shape == sample_submission.shape) # Vérification du format avec le format cible
    if is_valid:
        soumission.to_csv(f'{nom_sortie}.csv') # Génération du fichier de soumission selon le nom entré en paramètre
        print(f"Fichier '{nom_sortie}.csv' généré avec succès !") 
    else:
        print("Attention, les dimensions ne correspondent pas au fichier sample.")

In [ ]:
def benchmark(column):
    col = column.copy()
    col = col.interpolate(method='linear', limit_direction='both')
    return col

In [ ]:
def echantilloner (nb_echantillon):
    # Si none alors pas d'échantillonage
    holed_cols = [col for col in x_test.columns if 'holed' in col]
    complete_cols = [col for col in x_test.columns if 'holed' not in col]
    x_test_filled = x_test[holed_cols].copy()
    if nb_echantillon != None :
        X_features = x_test[complete_cols].sample(n=nb_echantillon, axis=1, random_state=67) # Permet de faire un échantillon parmis toutes les données pour accélérer le calcul (le calcul de base prend environ 1h sans échantillon)
    else :
        X_features = x_test[complete_cols]
    return holed_cols, x_test_filled, X_features

### Soumission de base (interpolation linéaire)

Fonction d'interpolation linéaire (remplissage des trous par une ligne droite, comme lors des précédents TP, pour éviter une erreur nan)

In [ ]:
def interpolation_lineaire(column):
    return column.interpolate(method='linear', limit_direction='both') # Evite les erreurs nan comme lors des derniers TP

In [ ]:
holed_cols_test = [col for col in x_test.columns if 'holed' in col]
y_pred_test = x_test[holed_cols_test].apply(interpolation_lineaire, axis=0)
submission = y_pred_test.loc[sample_submission.index, sample_submission.columns]

generer_soumission(submission, "interpolation_lineaire")

### Régression linéaire
Le score donné par cette regression linéaire est de 93 (mais nécessite les données complètes et ça tourne pendant un peu plus d'une heure)

Avec un échantillon de 6000 et un head à 30 on obtient un score de 98 avec un temps d'exécution de 15 min environ

In [ ]:
def regression_lineaire():

    holed_cols, x_test_filled, X_features_reduced = echantilloner(None)
    print("Lancement de la régression linéaire sur l'échantillon...")

    for target_col in tqdm(holed_cols):
        target_series = x_test[target_col]
        
        mask_known = target_series.notna()
        mask_missing = target_series.isna()
        
        if mask_known.sum() < 2 or not mask_missing.any():
            continue
            
        # Enlève les messages d'erreur inutiles
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            corrs = X_features_reduced[mask_known].corrwith(target_series[mask_known]).abs()
        
        corrs = corrs.fillna(0)
        top_vars = corrs.sort_values(ascending=False).head(50).index
        
        model = LinearRegression()
        model.fit(X_features_reduced.loc[mask_known, top_vars], target_series[mask_known])        
        predictions = model.predict(X_features_reduced.loc[mask_missing, top_vars])
        x_test_filled.loc[mask_missing, target_col] = predictions

    submission = x_test_filled.loc[sample_submission.index, sample_submission.columns]
    generer_soumission(submission, "regression_lineaire_full")
    return submission

regression_lineaire()



### Arbres

Fonctionne moins bien qu'une régression linéaire (106 contre 103 pour 2000 échantillons)

In [ ]:
def regression_arbre_decision():
    holed_cols, x_test_filled, X_features_reduced = echantilloner(2000)

    print("Lancement de l'Arbre de Décision (Sécurité anti-valeurs aberrantes)...")

    for target_col in tqdm(holed_cols):
        target_series = x_test[target_col]
        
        mask_known = target_series.notna()
        mask_missing = target_series.isna()
        
        if mask_known.sum() < 2 or not mask_missing.any():
            continue
            
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            corrs = X_features_reduced[mask_known].corrwith(target_series[mask_known]).abs()
        
        corrs = corrs.fillna(0)
        top_vars = corrs.sort_values(ascending=False).head(15).index
        
        # Arbre max_depth = 6
        model = DecisionTreeRegressor(max_depth=6, random_state=42)
        
        model.fit(X_features_reduced.loc[mask_known, top_vars], target_series[mask_known])
        
        predictions = model.predict(X_features_reduced.loc[mask_missing, top_vars])
        x_test_filled.loc[mask_missing, target_col] = predictions

    submission = x_test_filled.loc[sample_submission.index, sample_submission.columns]
    
    generer_soumission(submission, "arbre_decision_depth6")
    
    return submission

ma_soumission_arbre = regression_arbre_decision()

### Réseau de neurone
Version basée sur le dernier TP avec Tensorflow, elle n'est pas bien optimisée pour le texte et obtient un score de 94 pour 1h30 de calcul

In [ ]:
def regression_reseau_neurones_opti():
    
    holed_cols, x_test_filled, X_features_reduced = echantilloner(6000)

    print("Lancement du Réseau de Neurones...")
    
    tf.get_logger().setLevel('ERROR')

    for target_col in tqdm(holed_cols):
        target_series = x_test[target_col]
        
        mask_known = target_series.notna()
        mask_missing = target_series.isna()
        
        if mask_known.sum() < 2 or not mask_missing.any():
            continue
            
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            corrs = X_features_reduced[mask_known].corrwith(target_series[mask_known]).abs()
        
        corrs = corrs.fillna(0)
        
        top_vars = corrs.sort_values(ascending=False).head(50).index
        
        X_train_raw = X_features_reduced.loc[mask_known, top_vars].values
        y_train_nn = target_series[mask_known].values
        X_missing_raw = X_features_reduced.loc[mask_missing, top_vars].values
        
        scaler = StandardScaler() # Normalisation
        X_train_nn = scaler.fit_transform(X_train_raw)
        X_missing_nn = scaler.transform(X_missing_raw)
        # -----------------------------------------
        
        model = Sequential([Dense(64, activation='relu', input_shape=(len(top_vars),)), Dense(32, activation='relu'), Dense(1)])
        
        model.compile(optimizer='adam', loss='mae')
        
        early_stop = EarlyStopping(monitor='loss', patience=10, verbose=0) # Augmentation de la patiente par rapport à la dernière itération
        
        model.fit(X_train_nn, y_train_nn, epochs=100, batch_size=32, verbose=0, callbacks=[early_stop])
        
        predictions = model.predict(X_missing_nn, verbose=0)
        
        x_test_filled.loc[mask_missing, target_col] = predictions.flatten()

    submission = x_test_filled.loc[sample_submission.index, sample_submission.columns]
    
    generer_soumission(submission, "reseau_neurones_keras_tensorflow")
    
    return submission

ma_soumission_finale = regression_reseau_neurones_opti()

Version utilisant Sklearn, plus adaptée pour du texte et le contexte du hackaton. Score de 83 en 25 min de calcul

In [ ]:

def regression_gradient_boosting_features():
    
    holed_cols, x_test_filled, X_features_reduced = echantilloner(6000) # A modifier pour affinement

    print("Lancement du Gradient Boosting (Arbres) + Feature Engineering...")

    for target_col in tqdm(holed_cols):
        target_series = x_test[target_col]
        
        mask_known = target_series.notna()
        mask_missing = target_series.isna()
        
        if mask_known.sum() < 2 or not mask_missing.any():
            continue
            
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            corrs = X_features_reduced[mask_known].corrwith(target_series[mask_known]).abs()
        
        corrs = corrs.fillna(0)
        
        # 50 meilleures courbes (paramètre à modifier pour afiner)
        top_vars = corrs.sort_values(ascending=False).head(50).index
        
        # Extraction des données brutes
        X_train_raw = X_features_reduced.loc[mask_known, top_vars].values
        y_train = target_series[mask_known].values
        X_missing_raw = X_features_reduced.loc[mask_missing, top_vars].values
        
        # Création de nouvelles colones pour optimiser l'algo
        def ajouter_features_statistiques(X):
            moyenne = np.mean(X, axis=1, keepdims=True)
            ecart_type = np.std(X, axis=1, keepdims=True)
            maximum = np.max(X, axis=1, keepdims=True)
            minimum = np.min(X, axis=1, keepdims=True)
            # Collage des nouvelles colonnes avec les 50 courbes
            return np.hstack((X, moyenne, ecart_type, maximum, minimum))
            
        X_train_enriched = ajouter_features_statistiques(X_train_raw)
        X_missing_enriched = ajouter_features_statistiques(X_missing_raw)
        
        model = HistGradientBoostingRegressor(
            loss='absolute_error', # Minimise l'erreur
            max_iter=400,         
            learning_rate=0.05,    
            max_depth=5, # Arbres pas trop profonds
            random_state=67
        )
        
        model.fit(X_train_enriched, y_train)
        predictions = model.predict(X_missing_enriched)
        x_test_filled.loc[mask_missing, target_col] = predictions

    submission = x_test_filled.loc[sample_submission.index, sample_submission.columns]
    generer_soumission(submission, "reseau_neurones_sklearn")
    
    return submission

ma_soumission_elite = regression_gradient_boosting_features()